In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "/root/.cache/modelscope/hub/models/Xunzillm4cc/Xunzi-Qwen1.5-7B"

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    trust_remote_code=True,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()

test_sentence = "学而时习之不亦乐乎"

messages = [
    {
        "role": "system", 
        "content": "你是一个文言文分词工具。请直接输出空格分词后的结果，严禁输出任何翻译、解释、标点说明或多余内容。"
    },
    {
        "role": "user", 
        "content": f"输入：学而时习之不亦说乎\n输出：学 而 时习 之 不亦 说 乎\n\n现在请分词：\n输入：{test_sentence}\n输出："
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,       
        temperature=0.2,             
        do_sample=False,
        repetition_penalty=1.2,     
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

# 只取新生成的部分
generated = outputs[0][inputs.input_ids.shape[1]:]
result = tokenizer.decode(generated, skip_special_tokens=True).strip()

# 进一步清理：如果模型输出包含“回答”或换行，强制截断
result = result.split('\n')[0].split('回答')[0].strip()

print(f"原文：{test_sentence}")
print(f"分词：{result}")